# Labwork 3: Clustering and K-Means on Subspace

This notebook implements the lab in a simple way.

Main idea: clustering does **not** use labels during training. The labels are used only after clustering to evaluate and comment on the results.

We will:

1. Select two labeled datasets.
2. Run **K-Means** and **Agglomerative Hierarchical Clustering (AHC)** on the original data.
3. Reduce the data with **PCA**, **SVD**, and **t-SNE**.
4. Run the same two clustering models on the reduced data.
5. Compare metrics and comment on every model result.
6. Visualize clusters in the reduced spaces.


## 1. Setup


In [ ]:
import os
import warnings

os.environ["LOKY_MAX_CPU_COUNT"] = str(max(1, min(4, (os.cpu_count() or 2) - 1)))
warnings.filterwarnings("ignore", message="Could not find the number of physical cores.*")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.metrics import (
    adjusted_rand_score,
    completeness_score,
    homogeneity_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
pd.set_option("display.precision", 4)
plt.style.use("default")


## 2. Select Two Labeled Datasets

The two datasets are from scikit-learn:

- **Wine**: 178 samples, 13 numeric features, 3 classes.
- **Breast Cancer Wisconsin**: 569 samples, 30 numeric features, 2 classes.

The target labels are kept for evaluation only.


In [ ]:
datasets = {
    "Wine": load_wine(as_frame=True),
    "Breast Cancer": load_breast_cancer(as_frame=True),
}

dataset_summary = []
for dataset_name, dataset in datasets.items():
    dataset_summary.append({
        "Dataset": dataset_name,
        "Samples": dataset.data.shape[0],
        "Features": dataset.data.shape[1],
        "Classes": len(np.unique(dataset.target)),
    })

pd.DataFrame(dataset_summary)


## 3. Simple Helper Functions

Higher ARI, NMI, homogeneity, completeness, and silhouette are better.

- **ARI/NMI/Homogeneity/Completeness** compare clusters with the true labels.
- **Silhouette** uses only the feature space and does not need labels.


In [ ]:
def evaluate(x, y_true, y_pred):
    return {
        "ARI": adjusted_rand_score(y_true, y_pred),
        "NMI": normalized_mutual_info_score(y_true, y_pred),
        "Homogeneity": homogeneity_score(y_true, y_pred),
        "Completeness": completeness_score(y_true, y_pred),
        "Silhouette": silhouette_score(x, y_pred),
    }


def run_two_models(x, y_true, n_clusters):
    models = {
        "K-Means": KMeans(n_clusters=n_clusters, n_init=20, random_state=RANDOM_STATE),
        "AHC": AgglomerativeClustering(n_clusters=n_clusters, linkage="ward"),
    }

    rows = []
    predictions = {}
    for model_name, model in models.items():
        y_pred = model.fit_predict(x)
        predictions[model_name] = y_pred
        rows.append({"Model": model_name, **evaluate(x, y_true, y_pred)})

    return rows, predictions


def reduce_to_2d(x):
    return {
        "PCA": PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(x),
        "SVD": TruncatedSVD(n_components=2, random_state=RANDOM_STATE).fit_transform(x),
        "t-SNE": TSNE(
            n_components=2,
            random_state=RANDOM_STATE,
            init="pca",
            learning_rate="auto",
            perplexity=30,
        ).fit_transform(x),
    }


def quality_label(value):
    if value >= 0.75:
        return "strong"
    if value >= 0.50:
        return "moderate"
    if value >= 0.25:
        return "weak"
    return "poor"


def comment_row(row, dataset_best_ari):
    model = row["Model"]
    space = row["Space"]
    ari = row["ARI"]
    nmi = row["NMI"]
    sil = row["Silhouette"]

    if ari == dataset_best_ari:
        comparison = "This is the best ARI result for this dataset."
    elif ari >= dataset_best_ari - 0.05:
        comparison = "This is close to the best result for this dataset."
    else:
        comparison = "This is lower than the best result, so the clusters match the labels less well."

    return (
        f"- **{model} on {space}**: ARI={ari:.3f}, NMI={nmi:.3f}, "
        f"Silhouette={sil:.3f}. The label match is **{quality_label(ari)}**. {comparison}"
    )


## 4. Run Clustering on Original and Reduced Data

For each dataset, the features are standardized first. Then K-Means and AHC are run on:

- original scaled data
- PCA 2D data
- SVD 2D data
- t-SNE 2D data


In [ ]:
all_results = []
embeddings = {}
predictions = {}
true_labels = {}

for dataset_name, dataset in datasets.items():
    x = dataset.data.to_numpy()
    y = dataset.target.to_numpy()
    n_clusters = len(np.unique(y))
    x_scaled = StandardScaler().fit_transform(x)

    true_labels[dataset_name] = y
    embeddings[dataset_name] = {"Original": x_scaled}

    original_rows, original_predictions = run_two_models(x_scaled, y, n_clusters)
    predictions[(dataset_name, "Original")] = original_predictions

    for row in original_rows:
        all_results.append({"Dataset": dataset_name, "Space": "Original", **row})

    for space_name, x_reduced in reduce_to_2d(x_scaled).items():
        embeddings[dataset_name][space_name] = x_reduced
        reduced_rows, reduced_predictions = run_two_models(x_reduced, y, n_clusters)
        predictions[(dataset_name, space_name)] = reduced_predictions

        for row in reduced_rows:
            all_results.append({"Dataset": dataset_name, "Space": space_name, **row})

results = pd.DataFrame(all_results)
results = results[["Dataset", "Space", "Model", "ARI", "NMI", "Homogeneity", "Completeness", "Silhouette"]]
results.sort_values(["Dataset", "Space", "Model"]).reset_index(drop=True)


## 5. Comments on Original Clustering Models

This section focuses on the first lab requirement: run AHC and K-Means on the original data and comment on each model.


In [ ]:
original_results = results[results["Space"] == "Original"].sort_values(["Dataset", "Model"]).reset_index(drop=True)
original_results


In [ ]:
comment_lines = []
for dataset_name in datasets:
    dataset_rows = results[results["Dataset"] == dataset_name]
    best_ari = dataset_rows["ARI"].max()
    comment_lines.append(f"### {dataset_name}: original data")
    for _, row in original_results[original_results["Dataset"] == dataset_name].iterrows():
        comment_lines.append(comment_row(row, best_ari))
    comment_lines.append("")

display(Markdown("\n".join(comment_lines)))


## 6. Comments on K-Means and AHC After Dimensionality Reduction

This section focuses on the second lab requirement: run the same models in reduced subspaces and comment on each result.


In [ ]:
reduced_results = results[results["Space"] != "Original"].sort_values(["Dataset", "Space", "Model"]).reset_index(drop=True)
reduced_results


In [ ]:
comment_lines = []
for dataset_name in datasets:
    dataset_rows = results[results["Dataset"] == dataset_name]
    best_ari = dataset_rows["ARI"].max()
    comment_lines.append(f"### {dataset_name}: reduced spaces")
    for _, row in reduced_results[reduced_results["Dataset"] == dataset_name].iterrows():
        comment_lines.append(comment_row(row, best_ari))
    comment_lines.append("")

display(Markdown("\n".join(comment_lines)))


## 7. Compare Original Data with Reduced Data

For each dataset and model, compare the original result with the best reduced-space result.


In [ ]:
original_baseline = results[results["Space"] == "Original"].copy()
original_baseline["Type"] = "Original"

best_reduced = (
    results[results["Space"] != "Original"]
    .sort_values("ARI", ascending=False)
    .groupby(["Dataset", "Model"], as_index=False)
    .head(1)
    .copy()
)
best_reduced["Type"] = "Best reduced"

comparison = pd.concat([original_baseline, best_reduced], ignore_index=True)
comparison = comparison[["Dataset", "Model", "Type", "Space", "ARI", "NMI", "Silhouette"]]
comparison.sort_values(["Dataset", "Model", "Type"]).reset_index(drop=True)


In [ ]:
comparison_comments = []
for dataset_name in datasets:
    comparison_comments.append(f"### {dataset_name}: original vs reduced")
    for model_name in ["K-Means", "AHC"]:
        original = comparison[(comparison["Dataset"] == dataset_name) & (comparison["Model"] == model_name) & (comparison["Type"] == "Original")].iloc[0]
        reduced = comparison[(comparison["Dataset"] == dataset_name) & (comparison["Model"] == model_name) & (comparison["Type"] == "Best reduced")].iloc[0]
        delta = reduced["ARI"] - original["ARI"]
        if delta > 0.02:
            message = f"best reduced space improves ARI by {delta:.3f}."
        elif delta < -0.02:
            message = f"original space is better by {-delta:.3f} ARI."
        else:
            message = "original and reduced spaces perform almost the same."
        comparison_comments.append(
            f"- **{model_name}**: original ARI={original['ARI']:.3f}; best reduced is "
            f"{reduced['Space']} with ARI={reduced['ARI']:.3f}. {message}"
        )
    comparison_comments.append("")

display(Markdown("\n".join(comparison_comments)))


## 8. Visualize Clusters in Reduced Space

Each figure shows the true labels, K-Means clusters, and AHC clusters in a 2D reduced space.


In [ ]:
def plot_reduced_space(dataset_name, space_name):
    x_2d = embeddings[dataset_name][space_name]
    y = true_labels[dataset_name]
    preds = predictions[(dataset_name, space_name)]

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
    panels = [
        ("True labels", y),
        ("K-Means", preds["K-Means"]),
        ("AHC", preds["AHC"]),
    ]

    for ax, (title, labels) in zip(axes, panels):
        ax.scatter(x_2d[:, 0], x_2d[:, 1], c=labels, cmap="tab10", s=22, alpha=0.85)
        ax.set_title(title)
        ax.set_xlabel("Component 1")
        ax.set_ylabel("Component 2")
        ax.grid(alpha=0.3)

    fig.suptitle(f"{dataset_name} - {space_name}")
    plt.show()


for dataset_name in datasets:
    for space_name in ["PCA", "SVD", "t-SNE"]:
        plot_reduced_space(dataset_name, space_name)


## 9. Final Conclusion

K-Means and AHC can both cluster unlabeled feature data. The true labels are only used after fitting to measure how well clusters match the known classes.

The comments above are the most important part of this labwork. They show that clustering quality depends on both the dataset and the feature space. A dimensionality reduction method can improve a model when it removes noise or makes groups more compact, but it can also reduce performance if useful information is lost.
